# FreeSurfer 1: prepare a standard dataset

We have a series of notebooks to demonstrate some standard analyses with FreeSurfer.  
This notebooks shows how to prepare a dataset for analysis.

We will use a clinical sample from OpenNeuro: ["T1-weighted structural MRI study of cannabis users at baseline and 3 years follow up"](https://openneuro.org/datasets/ds000174/versions/1.0.1).  
The dataset includes T1 scans from cannabis users and healthy controls at a baseline timepoint, and at 3-year follow-up.  

We will load the data, extract FreeSurfer cortical thickness and subcortical volume estimates, and create multiple tables with only cross-sectional, only longitudinal, and combined data.

In [115]:
# general imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# monkey patch to make progress bars show as ASCII instead of as widgets
import tqdm.notebook
tqdm.notebook.tqdm = tqdm.tqdm
tqdm = tqdm.notebook.tqdm

In [116]:
# load local nispace, for testing
# COMMENT THIS OUT IF YOU RUN THIS LOCALLY AFTER INSTALLING NISPACE
import sys
sys.path.append("/Users/llotter/projects/nispace/")

## Data preparation: Download data from an OpenNeuro dataset

We will use a dataset of cnnabis users and healthy controls originally hosted on OpenNeuro und also used by [Andy's BrainBook](https://andysbrainbook.readthedocs.io/en/latest/FreeSurfer/FS_ShortCourse/FS_02_DownloadInstall.html).  
We provide a version of the dataset including FreeSurfer 8.1.0 derivatives via GIN: https://gin.g-node.org/llotter/ds000174-freesurfer  
The GIN dataset can conveniently be cloned with datalad. To do that locally, you will first have to [install datalad](https://handbook.datalad.org/en/latest/intro/installation.html).

In [117]:
# clone the dataset
!datalad clone git@gin.g-node.org:/llotter/ds000174-freesurfer.git

We will only use FreeSurfer ´".stats"` files for our analyses. These are contained in each FreeSurfer output folder within the "stats" directory.  
We do not have to download the whole dataset; datalad allows to selectively download folders using wildcards.

In [118]:
# download the stats folder of each subject/session
!datalad get ds000174-freesurfer/derivatives/freesurfer/sub-*/stats

action summary:
  get (notneeded: 84)


We define the dataset and freesurfer directories for further use

In [119]:
dataset_dir = Path.cwd() / "ds000174-freesurfer"
freesurfer_dir = dataset_dir / "derivatives" / "freesurfer"
assert dataset_dir.exists() and freesurfer_dir.exists()

#### Dataset README

This is a BIDS dataset, so there is a README file.

In [120]:
with open(dataset_dir / "README") as f:
    print("".join(f.readlines()))

This dataset was obtained from the OpenfMRI project (http://www.openfmri.org).

Accession #: ds000174
Description: T1-weighted structural MRI study of cannabis users at baseline and 3 years follow up

The scores from the CUDIT (Cannabis Use Disorder Identification Test) and AUDIT
(Alcohol Use Disorder Identification Test) are included in the participants.tsv file.

This dataset is made available under the Creative Commons Attribution-NonCommercial 4.0
International license: http://creativecommons.org/licenses/by-nc/4.0/legalcode.txt





#### Subject characteristics

There also is a `participants.tsv` file that contains subject characteristics.

In [121]:
df_participants = pd.read_table(dataset_dir / "participants.tsv", index_col=0)

# add the "sub-" prefix to the participant index column
df_participants.index = "sub-" + df_participants.index.astype(str)

# This is what the dataframe looks like:
print("participants.tsv:")
display(df_participants.head(5))

# That's our two groups:
print("Group statistics:")
df_participants.describe()

participants.tsv:


,group,gender,age at baseline,age at onset first CB use,age at onset frequent CB use,cudit total baseline,cudit total follow-up,audit total baseline,audit total follow-up
participant_id,,,,,,,,,
sub-202,HC,female,25.62,18.0,NaN,1,0,6,3
sub-206,HC,male,18.55,NaN,NaN,0,1,2,9
sub-207,HC,female,17.66,17.0,NaN,0,1,3,11
sub-209,HC,male,19.75,14.0,NaN,0,0,11,13
sub-211,HC,male,22.20,21.0,NaN,0,0,0,4


Group statistics:


,age at baseline,age at onset first CB use,age at onset frequent CB use,cudit total baseline,cudit total follow-up,audit total baseline,audit total follow-up
count,42.000000,33.000000,20.000000,42.000000,42.000000,42.000000,42.000000
mean,21.066905,16.060606,16.200000,6.071429,6.404762,5.285714,7.285714
std,2.322583,2.970579,2.375312,7.816159,8.701162,3.452010,4.340876
min,17.660000,11.500000,13.000000,0.000000,0.000000,0.000000,0.000000
25%,19.460000,14.000000,14.000000,0.000000,0.000000,2.250000,4.000000
50%,20.440000,15.000000,16.000000,0.500000,1.000000,5.500000,7.000000
75%,22.672500,18.000000,18.000000,10.000000,9.750000,7.750000,10.000000
max,26.040000,23.000000,22.000000,27.000000,30.000000,12.000000,20.000000


#### Collect FreeSurfer data

For now, we will use standard FreeSurfer output from the cortical segmentation. The two standard parcellations are Desikan-Killiany and Destrieux.
We assume that you have FreeSurfer installed on your system and accessible from the command line, so we can use the dedicated FreeSurfer command `aparcstats2table` to gather segmentation output across multiple subjects.

We will work with the Desikan-Killiany parcellation: "aparc" (Destrieux would be: "aparc.a2009s").

As noted above, the dataset contains longitudinal data (sessions "BL" and "FU"). We will later split up the data into different tables according to the analyses we will do.

In [122]:
import os
import subprocess

# set FreeSurfer's SUBJECTS_DIR to our datasets FreeSurfer output directory
os.environ["SUBJECTS_DIR"] = str(freesurfer_dir)

# get all subject/session names (each subject has a baseline and a follow-up scan)
# skip some subjects because of broken data
# TODO: fix the broken subjects
subjects_drop = ["sub-122", "sub-303"]
subjects = [d.name for d in sorted(freesurfer_dir.glob("sub-*"))  
            if not any(s in d.name for s in subjects_drop)]

# settings:
parc = "aparc"
meas = "thickness"  # can be "thickness", "volume", or "area"

# we have to run the command separately for each hemisphere and parcellation
for hemi in ["lh", "rh"]:
    
    # this will be our output table file
    aparcstats_file = freesurfer_dir / f"stats_{parc}_{hemi}.tsv"

    # command
    cmd = [
        "aparcstats2table",
        "--subjects", " ".join(subjects),
        "--hemi", hemi,
        "--parc", parc,
        "--meas", meas,
        "--tablefile", str(aparcstats_file),
        "--common-parcs"
    ]
    cmd = " ".join(cmd)
    print(cmd)
    # run the command
    try:
        subprocess.run(cmd, shell=True, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error running command: {e}")

aparcstats2table --subjects sub-101_ses-BL sub-101_ses-FU sub-103_ses-BL sub-103_ses-FU sub-104_ses-BL sub-104_ses-FU sub-108_ses-BL sub-108_ses-FU sub-109_ses-BL sub-109_ses-FU sub-112_ses-BL sub-112_ses-FU sub-116_ses-BL sub-116_ses-FU sub-117_ses-BL sub-117_ses-FU sub-119_ses-BL sub-119_ses-FU sub-121_ses-BL sub-121_ses-FU sub-123_ses-BL sub-123_ses-FU sub-124_ses-BL sub-124_ses-FU sub-125_ses-BL sub-125_ses-FU sub-126_ses-BL sub-126_ses-FU sub-127_ses-BL sub-127_ses-FU sub-128_ses-BL sub-128_ses-FU sub-130_ses-BL sub-130_ses-FU sub-132_ses-BL sub-132_ses-FU sub-133_ses-BL sub-133_ses-FU sub-202_ses-BL sub-202_ses-FU sub-206_ses-BL sub-206_ses-FU sub-207_ses-BL sub-207_ses-FU sub-209_ses-BL sub-209_ses-FU sub-211_ses-BL sub-211_ses-FU sub-213_ses-BL sub-213_ses-FU sub-218_ses-BL sub-218_ses-FU sub-222_ses-BL sub-222_ses-FU sub-302_ses-BL sub-302_ses-FU sub-304_ses-BL sub-304_ses-FU sub-305_ses-BL sub-305_ses-FU sub-306_ses-BL sub-306_ses-FU sub-308_ses-BL sub-308_ses-FU sub-309_ses-

#### Get subcortical volumes and Euler number as a quality covariate

Subcortical volumes are stort in "aseg.stats" files.  
The Euler number is quite a good quality measure for structural MRI scans. FreeSurfer outputs it only via the `asegstats2table` command.

In [123]:
# this will be our output table file
asegstats_file = freesurfer_dir / "stats_aseg.tsv"

# assemble the command 
cmd = [
    "asegstats2table",
    "--subjects", " ".join(subjects),
    "--tablefile", str(asegstats_file),
    "--common-segs"
]
cmd = " ".join(cmd)
print(cmd)

# run the command
subprocess.run(cmd, shell=True)

asegstats2table --subjects sub-101_ses-BL sub-101_ses-FU sub-103_ses-BL sub-103_ses-FU sub-104_ses-BL sub-104_ses-FU sub-108_ses-BL sub-108_ses-FU sub-109_ses-BL sub-109_ses-FU sub-112_ses-BL sub-112_ses-FU sub-116_ses-BL sub-116_ses-FU sub-117_ses-BL sub-117_ses-FU sub-119_ses-BL sub-119_ses-FU sub-121_ses-BL sub-121_ses-FU sub-123_ses-BL sub-123_ses-FU sub-124_ses-BL sub-124_ses-FU sub-125_ses-BL sub-125_ses-FU sub-126_ses-BL sub-126_ses-FU sub-127_ses-BL sub-127_ses-FU sub-128_ses-BL sub-128_ses-FU sub-130_ses-BL sub-130_ses-FU sub-132_ses-BL sub-132_ses-FU sub-133_ses-BL sub-133_ses-FU sub-202_ses-BL sub-202_ses-FU sub-206_ses-BL sub-206_ses-FU sub-207_ses-BL sub-207_ses-FU sub-209_ses-BL sub-209_ses-FU sub-211_ses-BL sub-211_ses-FU sub-213_ses-BL sub-213_ses-FU sub-218_ses-BL sub-218_ses-FU sub-222_ses-BL sub-222_ses-FU sub-302_ses-BL sub-302_ses-FU sub-304_ses-BL sub-304_ses-FU sub-305_ses-BL sub-305_ses-FU sub-306_ses-BL sub-306_ses-FU sub-308_ses-BL sub-308_ses-FU sub-309_ses-B

CompletedProcess(args='asegstats2table --subjects sub-101_ses-BL sub-101_ses-FU sub-103_ses-BL sub-103_ses-FU sub-104_ses-BL sub-104_ses-FU sub-108_ses-BL sub-108_ses-FU sub-109_ses-BL sub-109_ses-FU sub-112_ses-BL sub-112_ses-FU sub-116_ses-BL sub-116_ses-FU sub-117_ses-BL sub-117_ses-FU sub-119_ses-BL sub-119_ses-FU sub-121_ses-BL sub-121_ses-FU sub-123_ses-BL sub-123_ses-FU sub-124_ses-BL sub-124_ses-FU sub-125_ses-BL sub-125_ses-FU sub-126_ses-BL sub-126_ses-FU sub-127_ses-BL sub-127_ses-FU sub-128_ses-BL sub-128_ses-FU sub-130_ses-BL sub-130_ses-FU sub-132_ses-BL sub-132_ses-FU sub-133_ses-BL sub-133_ses-FU sub-202_ses-BL sub-202_ses-FU sub-206_ses-BL sub-206_ses-FU sub-207_ses-BL sub-207_ses-FU sub-209_ses-BL sub-209_ses-FU sub-211_ses-BL sub-211_ses-FU sub-213_ses-BL sub-213_ses-FU sub-218_ses-BL sub-218_ses-FU sub-222_ses-BL sub-222_ses-FU sub-302_ses-BL sub-302_ses-FU sub-304_ses-BL sub-304_ses-FU sub-305_ses-BL sub-305_ses-FU sub-306_ses-BL sub-306_ses-FU sub-308_ses-BL sub-3

### Load FreeSurfer stats tables

Now we load the FreeSurfer stats tables into pandas dataframes. We will combine left and right hemisphere data into one dataframe (L -> R). This is the way NiSpace works with bilateral surface data.  
We will load the data in the Destrieux parcellation (aparc.a2009s). Drop the ".a2009s" from the lines below to work with the DesikanKilliany data.

In [124]:
# load left and right hemisphere data
df_lh = pd.read_table(freesurfer_dir / f"stats_{parc}_lh.tsv", index_col=0)
df_rh = pd.read_table(freesurfer_dir / f"stats_{parc}_rh.tsv", index_col=0)
print("left hemisphere: shape:", df_lh.shape)

# drop the unnecessary columns
df_lh = df_lh.loc[:, ~df_lh.columns.isin(["lh_MeanThickness_thickness", "lh_WhiteSurfArea_area", "BrainSegVolNotVent", "eTIV"])]
df_rh = df_rh.loc[:, ~df_rh.columns.isin(["rh_MeanThickness_thickness", "rh_WhiteSurfArea_area", "BrainSegVolNotVent", "eTIV"])]
print("shape after dropping columns:", df_lh.shape)

# concatenate left and right hemisphere
df_cortex = pd.concat([df_lh, df_rh], axis=1)
print("Shape after concatenation:", df_cortex.shape)
display(df_cortex)

left hemisphere: shape: (80, 37)
shape after dropping columns: (80, 34)
Shape after concatenation: (80, 68)


,lh_bankssts_thickness,lh_caudalanteriorcingulate_thickness,lh_caudalmiddlefrontal_thickness,lh_cuneus_thickness,lh_entorhinal_thickness,lh_fusiform_thickness,lh_inferiorparietal_thickness,lh_inferiortemporal_thickness,lh_isthmuscingulate_thickness,lh_lateraloccipital_thickness,...,rh_rostralanteriorcingulate_thickness,rh_rostralmiddlefrontal_thickness,rh_superiorfrontal_thickness,rh_superiorparietal_thickness,rh_superiortemporal_thickness,rh_supramarginal_thickness,rh_frontalpole_thickness,rh_temporalpole_thickness,rh_transversetemporal_thickness,rh_insula_thickness
sub-101_ses-BL,2.490,2.749,2.218,1.915,3.216,2.769,2.204,2.944,2.476,2.086,...,2.978,2.204,2.520,1.912,2.963,2.426,2.542,3.729,2.328,3.140
sub-101_ses-FU,2.472,2.790,2.250,1.522,3.002,2.338,2.215,2.711,2.070,1.855,...,2.441,2.130,2.439,1.922,2.683,2.301,2.415,3.625,1.736,2.575
sub-103_ses-BL,2.490,2.693,2.327,2.146,3.032,2.740,2.382,2.733,2.530,2.184,...,2.538,2.305,2.594,1.977,2.942,2.444,3.041,3.238,2.433,2.913
sub-103_ses-FU,2.152,2.542,2.363,1.854,3.359,2.344,2.309,2.435,2.127,1.907,...,2.486,2.305,2.557,2.041,2.847,2.348,2.660,3.764,1.825,2.652
sub-104_ses-BL,2.207,2.402,2.213,1.989,3.258,2.730,2.477,2.546,2.414,2.204,...,2.346,2.161,2.581,2.127,2.659,2.565,2.626,3.076,2.695,2.767
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
sub-318_ses-FU,2.476,2.345,2.458,1.623,2.633,2.490,2.402,2.718,2.354,1.795,...,2.376,2.519,2.771,2.157,2.603,2.600,2.745,3.597,1.764,2.530
sub-319_ses-BL,2.938,2.555,2.513,2.079,3.082,3.001,2.429,3.009,2.433,2.261,...,2.902,2.334,2.725,2.046,3.155,2.584,2.855,3.615,2.682,3.220
sub-319_ses-FU,2.568,2.386,2.556,1.548,2.849,2.582,2.356,2.667,2.219,1.708,...,2.722,2.439,2.704,1.928,2.960,2.412,2.777,3.260,1.721,2.786
sub-320_ses-BL,2.438,2.224,2.294,1.949,3.627,2.924,2.257,2.869,2.273,2.180,...,2.665,2.262,2.739,2.015,3.053,2.563,2.330,3.807,2.110,3.060


#### Subcortical volumes, Euler number and eTIV as covariates

Euler number is stored as "SurfaceHoles". We will furthermore extract total intracranial volume (eTIV) data.

In [125]:
df_aseg = pd.read_table(freesurfer_dir / "stats_aseg.tsv", index_col=0)

# data that can be interesting for QC and as covariates are intracranial volume and the euler number
df_covariates = df_aseg[["SurfaceHoles", "EstimatedTotalIntraCranialVol"]]
print("Covariates")
display(df_covariates.head())

# subcortical data is contained in here but also ventricle volumes and whole-hemisphere volumes
# we are only interested in the actual subcortical volumes
df_subcortex = df_aseg[[
    'Left-Thalamus', 
    'Left-Caudate',
    'Left-Putamen',
    'Left-Pallidum',
    'Left-Hippocampus',
    'Left-Amygdala',
    'Left-Accumbens-area',
    'Left-VentralDC',
    'Right-Thalamus',
    'Right-Caudate',
    'Right-Putamen',
    'Right-Pallidum',
    'Right-Hippocampus',
    'Right-Amygdala',
    'Right-Accumbens-area',
    'Right-VentralDC'
]]
print("Subcortical volumes")
display(df_subcortex.head())


Covariates


,SurfaceHoles,EstimatedTotalIntraCranialVol
Measure:volume,,
sub-101_ses-BL,16.0,1.562223e+06
sub-101_ses-FU,55.0,1.585093e+06
sub-103_ses-BL,42.0,1.692744e+06
sub-103_ses-FU,76.0,1.715033e+06
sub-104_ses-BL,35.0,1.384595e+06


Subcortical volumes


,Left-Thalamus,Left-Caudate,Left-Putamen,Left-Pallidum,Left-Hippocampus,Left-Amygdala,Left-Accumbens-area,Left-VentralDC,Right-Thalamus,Right-Caudate,Right-Putamen,Right-Pallidum,Right-Hippocampus,Right-Amygdala,Right-Accumbens-area,Right-VentralDC
Measure:volume,,,,,,,,,,,,,,,,
sub-101_ses-BL,6846.9,3478.6,4986.4,1745.3,4099.6,1615.2,616.1,3820.1,6791.5,3557.0,4870.0,1554.6,3961.8,1554.1,532.7,3840.5
sub-101_ses-FU,7240.6,3583.9,4916.2,1477.4,4083.6,1480.6,484.0,4188.5,6971.6,3598.7,4923.0,1374.9,4042.9,1728.5,478.6,4256.9
sub-103_ses-BL,6924.6,4085.2,6231.7,1961.2,4525.7,1773.1,653.4,3708.9,6483.6,4210.4,5928.0,1790.1,4505.2,1793.7,607.8,3848.0
sub-103_ses-FU,7703.9,4032.7,6161.9,1450.3,4414.3,1644.5,466.7,3893.6,6895.1,3994.7,5758.0,1698.7,4459.6,1878.5,497.2,3870.3
sub-104_ses-BL,6122.1,3528.7,5709.8,1691.3,3758.0,1330.1,569.3,3391.2,6017.1,3650.9,5413.9,1704.4,3769.3,1471.7,744.9,3574.6


## Save tables

We will save tables for different analyses conducted in separate notebooks. 

In [126]:
## Participants
df_participants.to_csv(Path.cwd() / "freesurfer_participants.csv")

# -----------------------------------------------------------------------------
## Baseline data from both CB and HC groups
# cortical thickness
df_ct_baseline = (
    df_cortex[df_cortex.index.str.contains("BL")] # select BL sessions
    .assign(subject=lambda x: x.index.str.split("_").str[0]) # remove the "ses-BL" suffix from the index
    .set_index("subject")
)
print("Baseline data, CB and HC: cortical thickness,", df_ct_baseline.shape)
display(df_ct_baseline.head())
# subcortical volume
df_sc_baseline = (
    df_subcortex[df_subcortex.index.str.contains("BL")] # select BL sessions
    .assign(subject=lambda x: x.index.str.split("_").str[0]) # create a new index without the BL suffix
    .set_index("subject") 
)
# covariates
df_cov_baseline = (
    df_covariates[df_covariates.index.str.contains("BL")] # select BL sessions
    .assign(subject=lambda x: x.index.str.split("_").str[0]) # create a new index without the BL suffix
    .set_index("subject") 
)
print("Baseline data, CB and HC: covariates,", df_cov_baseline.shape)
display(df_cov_baseline.head())
# save all
df_ct_baseline.to_csv(Path.cwd() / "freesurfer_ct_baseline.csv")
df_sc_baseline.to_csv(Path.cwd() / "freesurfer_sc_baseline.csv")
df_cov_baseline.to_csv(Path.cwd() / "freesurfer_cov_baseline.csv")

# -----------------------------------------------------------------------------
## All data from both groups
# cortical thickness
df_ct_all = (
    df_cortex
    .assign(subject=lambda x: x.index.str.split("_").str[0],
            session=lambda x: x.index.str.split("_").str[1]) # create new indices with sub and session
    .set_index(["subject", "session"])
)
print("All data, CB and HC: cortical thickness,", df_ct_all.shape)
display(df_ct_all.head())
# cortical thickness
df_cov_all = (
    df_covariates
    .assign(subject=lambda x: x.index.str.split("_").str[0],
            session=lambda x: x.index.str.split("_").str[1]) # create new indices with sub and session
    .set_index(["subject", "session"])
)
print("All data, CB and HC: covariates,", df_cov_all.shape)
display(df_cov_all.head())
# save
df_ct_all.to_csv(Path.cwd() / "freesurfer_ct_all.csv")
df_cov_all.to_csv(Path.cwd() / "freesurfer_cov_all.csv")

# -----------------------------------------------------------------------------
## Longitudinal data from only the HC group
# cortical thickness
df_ct_longitudinal = (
    df_ct_all
    .query(f"subject in {df_participants.loc[df_participants.group=='HC'].index.to_list()}")
)
print("Longitudinal data, HC only: cortical thickness,", df_ct_longitudinal.shape)
display(df_ct_longitudinal.head())
# covariates
df_cov_longitudinal = (
    df_cov_all
    .query(f"subject in {df_participants.loc[df_participants.group=='HC'].index.to_list()}")
)
print("Longitudinal data, HC only: covariates,", df_cov_longitudinal.shape)
display(df_cov_longitudinal.head())
# save
df_ct_longitudinal.to_csv(Path.cwd() / "freesurfer_ct_longitudinal.csv")
df_cov_longitudinal.to_csv(Path.cwd() / "freesurfer_cov_longitudinal.csv")

Baseline data, CB and HC: cortical thickness, (40, 68)


,lh_bankssts_thickness,lh_caudalanteriorcingulate_thickness,lh_caudalmiddlefrontal_thickness,lh_cuneus_thickness,lh_entorhinal_thickness,lh_fusiform_thickness,lh_inferiorparietal_thickness,lh_inferiortemporal_thickness,lh_isthmuscingulate_thickness,lh_lateraloccipital_thickness,...,rh_rostralanteriorcingulate_thickness,rh_rostralmiddlefrontal_thickness,rh_superiorfrontal_thickness,rh_superiorparietal_thickness,rh_superiortemporal_thickness,rh_supramarginal_thickness,rh_frontalpole_thickness,rh_temporalpole_thickness,rh_transversetemporal_thickness,rh_insula_thickness
subject,,,,,,,,,,,,,,,,,,,,,
sub-101,2.490,2.749,2.218,1.915,3.216,2.769,2.204,2.944,2.476,2.086,...,2.978,2.204,2.520,1.912,2.963,2.426,2.542,3.729,2.328,3.140
sub-103,2.490,2.693,2.327,2.146,3.032,2.740,2.382,2.733,2.530,2.184,...,2.538,2.305,2.594,1.977,2.942,2.444,3.041,3.238,2.433,2.913
sub-104,2.207,2.402,2.213,1.989,3.258,2.730,2.477,2.546,2.414,2.204,...,2.346,2.161,2.581,2.127,2.659,2.565,2.626,3.076,2.695,2.767
sub-108,2.703,2.347,2.335,1.974,2.887,2.828,2.488,2.788,2.776,2.189,...,2.871,2.342,2.601,2.155,3.131,2.573,2.659,3.833,2.360,3.084
sub-109,2.830,2.404,2.338,2.250,3.261,2.954,2.366,2.923,2.474,2.313,...,3.294,2.235,2.667,2.088,3.158,2.539,2.525,3.895,2.980,3.229


Baseline data, CB and HC: covariates, (40, 2)


,SurfaceHoles,EstimatedTotalIntraCranialVol
subject,,
sub-101,16.0,1.562223e+06
sub-103,42.0,1.692744e+06
sub-104,35.0,1.384595e+06
sub-108,53.0,1.661247e+06
sub-109,23.0,1.621123e+06


All data, CB and HC: cortical thickness, (80, 68)


...

SurfaceHoles  EstimatedTotalIntraCranialVol
subject session                                             
sub-101 ses-BL           16.0                   1.562223e+06
        ses-FU           55.0                   1.585093e+06
sub-103 ses-BL           42.0                   1.692744e+06
        ses-FU           76.0                   1.715033e+06
sub-104 ses-BL           35.0                   1.384595e+06

Longitudinal data, HC only: cortical thickness, (42, 68)


lh_bankssts_thickness  lh_caudalanteriorcingulate_thickness  \
subject session                                                                
sub-202 ses-BL                   2.489                                 2.723   
        ses-FU                   2.480                                 2.487   
sub-206 ses-BL                   2.725                                 2.313   
        ses-FU                   2.640                                 2.365   
sub-207 ses-BL                   2.895                                 2.625   

                 lh_caudalmiddlefrontal_thickness  lh_cuneus_thickness  \
subject session                                                          
sub-202 ses-BL                              2.288                1.821   
        ses-FU                              2.423                1.363   
sub-206 ses-BL                              2.376                2.031   
        ses-FU                              2.495                1.621   
sub-207 ses-BL                              2.494                1.912   

                 lh_entorhinal_thickness  lh_fusiform_thickness  \
subject session                                                   
sub-202 ses-BL                     3.057                  2.790   
        ses-FU                     2.635                  2.457   
sub-206 ses-BL                     3.254                  2.779   
        ses-FU                     2.852                  2.475   
sub-207 ses-BL                     3.148                  2.810   

                 lh_inferiorparietal_thickness  lh_inferiortemporal_thickness  \
subject session                                                                 
sub-202 ses-BL                           2.254                          2.643   
        ses-FU                           2.290                          2.600   
sub-206 ses-BL                           2.567                          2.845   
        ses-FU                           2.447                          2.553   
sub-207 ses-BL                           2.405                          2.840   

                 lh_isthmuscingulate_thickness  lh_lateraloccipital_thickness  \
subject session                                                                 
sub-202 ses-BL                           2.770                          2.023   
        ses-FU                           2.263                          1.872   
sub-206 ses-BL                           2.542                          2.216   
        ses-FU                           2.340                          1.832   
sub-207 ses-BL                           2.512                          2.249   

                 ...  rh_rostralanteriorcingulate_thickness  \
subject session  ...                                          
sub-202 ses-BL   ...                                  2.596   
        ses-FU   ...                                  2.216   
sub-206 ses-BL   ...                                  3.005   
        ses-FU   ...                                  2.648   
sub-207 ses-BL   ...                                  2.843   

                 rh_rostralmiddlefrontal_thickness  \
subject session                                      
sub-202 ses-BL                               2.237   
        ses-FU                               2.305   
sub-206 ses-BL                               2.250   
        ses-FU                               2.361   
sub-207 ses-BL                               2.288   

                 rh_superiorfrontal_thickness  rh_superiorparietal_thickness  \
subject session                                                                
sub-202 ses-BL                          2.643                          1.933   
        ses-FU                          2.640                          1.848   
sub-206 ses-BL                          2.533                          2.098   
        ses-FU                          2.647                          2.068   
sub-207 ses-BL                

Longitudinal data, HC only: covariates, (42, 2)


SurfaceHoles  EstimatedTotalIntraCranialVol
subject session                                             
sub-202 ses-BL           15.0                   1.542527e+06
        ses-FU           61.0                   1.552628e+06
sub-206 ses-BL           38.0                   1.672606e+06
        ses-FU           84.0                   1.667527e+06
sub-207 ses-BL           25.0                   1.555500e+06